# 14.1 提示设计 (Prompt Design)

提示设计决定了模型输出的质量和格式，是与大模型交互的核心技能。

本节涵盖：零样本提示、少样本提示、思维链提示

## 1. 零样本与少样本提示

**零样本提示（Zero-Shot）**：不提供示例，直接描述任务让模型完成。

**少样本提示（Few-Shot）**：提供几个示例，让模型理解任务格式和模式。

**关键原则**：
- 清晰具体地描述任务
- 提供输出格式要求
- 示例要覆盖不同情况
- 示例顺序影响输出（近期偏差）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class SimplePromptModel(nn.Module):
    def __init__(self, d=64, vocab_size=100, max_len=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d, 2, d * 4, batch_first=True), 1
        )
        self.head = nn.Linear(d, vocab_size)

    def forward(self, x):
        h = self.embed(x)
        h = self.transformer(h)
        return self.head(h)

model = SimplePromptModel()

zero_shot_prompt = torch.tensor([[10, 20, 30, 99]])
few_shot_prompt = torch.tensor([[10, 20, 50, 10, 21, 51, 10, 22, 52, 10, 23, 99]])

zero_logits = model(zero_shot_prompt)
few_logits = model(few_shot_prompt)

print('=== Prompt Design ===')
print(f'Zero-shot prompt: {zero_shot_prompt.tolist()}')
print(f'  -> Output shape: {zero_logits.shape}')
print(f'\nFew-shot prompt: {few_shot_prompt.tolist()}')
print(f'  -> Output shape: {few_logits.shape}')
print(f'  -> Contains 3 examples before the query')
print(f'\nKey: Few-shot provides pattern examples for the model to follow.')
print(f'More examples = better pattern recognition, but uses more context.')

## 2. 思维链提示 (Chain-of-Thought)

**目的**：引导模型展示推理过程，提升复杂任务的准确率

**基本原理**：通过在提示中加入逐步推理的示例，引导模型在给出答案前先展示推理步骤。

**CoT变体**：
- **标准CoT**：提供带推理步骤的示例
- **Zero-Shot CoT**：添加"Let's think step by step"
- **Self-Consistency**：多次采样取多数投票
- **Tree of Thought**：探索多条推理路径

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class CoTDemo(nn.Module):
    def __init__(self, d=64, vocab_size=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d)
        self.gru = nn.GRU(d, 32, batch_first=True)
        self.head = nn.Linear(32, vocab_size)

    def forward(self, x):
        h = self.embed(x)
        h, _ = self.gru(h)
        return self.head(h)

model = CoTDemo()

direct_prompt = torch.tensor([[5, 10, 99]])
cot_prompt = torch.tensor([[5, 10, 50, 51, 52, 53, 99]])

direct_out = model(direct_prompt)
cot_out = model(cot_prompt)

print('=== Chain-of-Thought Prompting ===')
print(f'Direct: Q -> A')
print(f'CoT:    Q -> Step1 -> Step2 -> Step3 -> A')
print(f'\nDirect prompt length: {direct_prompt.shape[1]}')
print(f'CoT prompt length: {cot_prompt.shape[1]} (includes reasoning steps)')
print(f'\nKey: CoT decomposes complex reasoning into intermediate steps.')
print(f'"Let\'s think step by step" enables zero-shot CoT.')
print(f'Self-consistency: sample multiple CoT paths, take majority vote.')